# Last 10 Bitcoin Price Predictions Comparison
This notebook compares the last 10 price predictions (separated by commas) from both the enhanced and base models.

## Install Required Libraries

In [ ]:
!pip install transformers datasets torch peft accelerate matplotlib seaborn pandas numpy

## Load Required Libraries and Models

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from datasets import load_dataset
import json
import re

# Load the enhanced model
base_model_id = './Qwen3-8B'
adapter_path = './my-awesome-model_final_bitcoin-enhanced-prediction-dataset-with-local-comprehensive-news-v2/checkpoint-400'

# Load the base model and tokenizer
base_qwen_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(
    adapter_path,
    trust_remote_code=True
)

base_qwen_tokenizer = tokenizer

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
base_qwen_model.resize_token_embeddings(len(tokenizer))

# Load the enhanced model with LoRA adapter
enhanced_model = PeftModel.from_pretrained(base_qwen_model, adapter_path)
enhanced_model.eval()

print("Models loaded successfully!")

## Load Test Dataset

In [ ]:
# Load the enhanced dataset
test_dataset = load_dataset('tahamajs/bitcoin-enhanced-prediction-dataset-with-local-comprehensive-news', split='train')
print(f"Loaded {len(test_dataset)} test samples")

## Utility Functions for Price Extraction

In [ ]:
def extract_trading_recommendation_from_text(text):
    """Extract structured trading recommendation with 10-day forecast from model output"""
    import json
    
    # Try to find JSON-like structure in the text
    json_pattern = r'\{[^{}]*"forecast_10d"[^{}]*\}'
    matches = re.findall(json_pattern, text, re.DOTALL)
    
    if matches:
        for match in matches:
            try:
                # Clean up the JSON string
                cleaned_json = match.strip()
                parsed = json.loads(cleaned_json)
                
                # Validate required fields
                if all(key in parsed for key in ['action', 'confidence', 'forecast_10d']):
                    return parsed
            except json.JSONDecodeError:
                continue
    
    # Fallback: try to extract forecast_10d array separately
    forecast_pattern = r'"forecast_10d"\s*:\s*\[([^\]]+)\]'
    forecast_match = re.search(forecast_pattern, text)
    
    if forecast_match:
        try:
            price_str = forecast_match.group(1)
            prices = [float(p.strip()) for p in price_str.split(',')]
            return {
                'action': 'UNKNOWN',
                'confidence': 0,
                'forecast_10d': prices,
                'stop_loss': None,
                'take_profit': None
            }
        except:
            pass
    
    # Last resort: extract comma-separated numbers
    price_pattern = r'(\d+(?:\.\d+)?(?:,\s*\d+(?:\.\d+)?)*)'  
    matches = re.findall(price_pattern, text)
    
    if matches:
        longest_match = max(matches, key=len)
        try:
            prices = [float(p.strip()) for p in longest_match.split(',')]
            return {
                'action': 'UNKNOWN',
                'confidence': 0,
                'forecast_10d': prices[-10:] if len(prices) >= 10 else prices,
                'stop_loss': None,
                'take_profit': None
            }
        except:
            pass
    
    return None

def extract_last_10_prices_from_text(text):
    """Extract the last 10 price predictions from model output (backward compatibility)"""
    recommendation = extract_trading_recommendation_from_text(text)
    if recommendation and 'forecast_10d' in recommendation:
        return recommendation['forecast_10d']
    return []

def extract_all_prices_from_text(text):
    """Extract all price predictions from model output (backward compatibility)"""
    return extract_last_10_prices_from_text(text)

def format_input_enhanced(example):
    """Format input for the enhanced model"""
    instruction = example.get('instruction', '')
    user_input = example.get('input', '')
    messages = [
        {'role': 'system', 'content': instruction},
        {'role': 'user', 'content': user_input}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def format_input_base(example):
    """Format input for base Qwen model"""
    instruction = example.get('instruction', '')
    user_input = example.get('input', '')
    
    bitcoin_instruction = """You are a Bitcoin investment advisor. Based on the provided market data and news, provide a trading recommendation in JSON format with:
- action: "BUY", "SELL", or "HOLD"
- confidence: confidence level (0-100)
- stop_loss: recommended stop loss price
- take_profit: recommended take profit price
- forecast_10d: array of 10 price predictions for the next 10 days

Example format:
{"action":"SELL","confidence":68,"stop_loss":9593.73,"take_profit":8370.01,"forecast_10d":[9174.91, 8277.01, 6955.27, 7754.00, 7621.30, 8265.59, 8736.98, 8621.90, 8129.97, 8926.57]}"""
    
    messages = [
        {'role': 'system', 'content': bitcoin_instruction},
        {'role': 'user', 'content': f"{instruction}\n\n{user_input}\n\nPlease provide your trading recommendation in the JSON format specified above."}
    ]
    return base_qwen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print("Utility functions loaded!")

## Generate Last 10 Price Predictions

In [ ]:
# Generate predictions for multiple samples
num_samples = 20  # Number of test samples to analyze
results = []

print(f"Generating last 10 price predictions for {num_samples} samples...\n")

for i in range(min(num_samples, len(test_dataset))):
    test_example = test_dataset[i]
    sample_result = {
        'sample_id': i,
        'enhanced_recommendation': None,
        'base_recommendation': None,
        'actual_recommendation': None,
        'enhanced_last_10': [],
        'base_last_10': [],
        'actual_last_10': [],
        'enhanced_full': [],
        'base_full': [],
        'actual_full': []
    }
    
    # Enhanced Model Prediction
    enhanced_text = format_input_enhanced(test_example)
    enhanced_inputs = tokenizer(enhanced_text, return_tensors='pt', truncation=True, max_length=2048)
    enhanced_inputs = {k: v.to(enhanced_model.device) for k, v in enhanced_inputs.items()}
    
    with torch.no_grad():
        enhanced_outputs = enhanced_model.generate(
            **enhanced_inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    enhanced_generated = tokenizer.decode(enhanced_outputs[0][enhanced_inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    
    # Base Model Prediction
    base_text = format_input_base(test_example)
    base_inputs = base_qwen_tokenizer(base_text, return_tensors='pt', truncation=True, max_length=2048)
    base_inputs = {k: v.to(base_qwen_model.device) for k, v in base_inputs.items()}
    
    with torch.no_grad():
        base_outputs = base_qwen_model.generate(
            **base_inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=base_qwen_tokenizer.eos_token_id
        )
    
    base_generated = base_qwen_tokenizer.decode(base_outputs[0][base_inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    
    # Extract trading recommendations and prices
    enhanced_recommendation = extract_trading_recommendation_from_text(enhanced_generated)
    base_recommendation = extract_trading_recommendation_from_text(base_generated)
    actual_recommendation = extract_trading_recommendation_from_text(test_example.get('output', ''))
    
    enhanced_prices_full = enhanced_recommendation['forecast_10d'] if enhanced_recommendation else []
    base_prices_full = base_recommendation['forecast_10d'] if base_recommendation else []
    actual_prices_full = actual_recommendation['forecast_10d'] if actual_recommendation else []
    
    enhanced_prices_last_10 = enhanced_prices_full[-10:] if len(enhanced_prices_full) >= 10 else enhanced_prices_full
    base_prices_last_10 = base_prices_full[-10:] if len(base_prices_full) >= 10 else base_prices_full
    actual_prices_last_10 = actual_prices_full[-10:] if len(actual_prices_full) >= 10 else actual_prices_full
    
    # Store results
    sample_result.update({
        'enhanced_recommendation': enhanced_recommendation,
        'base_recommendation': base_recommendation,
        'actual_recommendation': actual_recommendation,
        'enhanced_last_10': enhanced_prices_last_10,
        'base_last_10': base_prices_last_10,
        'actual_last_10': actual_prices_last_10,
        'enhanced_full': enhanced_prices_full,
        'base_full': base_prices_full,
        'actual_full': actual_prices_full,
        'enhanced_text': enhanced_generated,
        'base_text': base_generated
    })
    
    results.append(sample_result)
    
    # Print progress with trading recommendations
    print(f"Sample {i+1}:")
    print(f"  Enhanced - Action: {enhanced_recommendation.get('action', 'N/A') if enhanced_recommendation else 'N/A'}, "
          f"Confidence: {enhanced_recommendation.get('confidence', 'N/A') if enhanced_recommendation else 'N/A'}%, "
          f"Forecast: {enhanced_prices_last_10}")
    print(f"  Base - Action: {base_recommendation.get('action', 'N/A') if base_recommendation else 'N/A'}, "
          f"Confidence: {base_recommendation.get('confidence', 'N/A') if base_recommendation else 'N/A'}%, "
          f"Forecast: {base_prices_last_10}")
    print(f"  Actual - Action: {actual_recommendation.get('action', 'N/A') if actual_recommendation else 'N/A'}, "
          f"Confidence: {actual_recommendation.get('confidence', 'N/A') if actual_recommendation else 'N/A'}%, "
          f"Forecast: {actual_prices_last_10}")
    print(f"  Counts - Enhanced: {len(enhanced_prices_full)}, Base: {len(base_prices_full)}, Actual: {len(actual_prices_full)}")
    print("-" * 80)

print(f"\n✅ Generated predictions for {len(results)} samples!")

## Trading Recommendations Analysis

In [ ]:
# Analyze trading recommendations
enhanced_recommendations = []
base_recommendations = []
actual_recommendations = []

for result in results:
    if result['enhanced_recommendation']:
        enhanced_recommendations.append(result['enhanced_recommendation'])
    if result['base_recommendation']:
        base_recommendations.append(result['base_recommendation'])
    if result['actual_recommendation']:
        actual_recommendations.append(result['actual_recommendation'])

print("=" * 80)
print("📊 TRADING RECOMMENDATIONS ANALYSIS")
print("=" * 80)

# Action distribution
def analyze_actions(recommendations, model_name):
    actions = [rec.get('action', 'UNKNOWN') for rec in recommendations]
    action_counts = {action: actions.count(action) for action in set(actions)}
    
    print(f"\n🎯 {model_name} MODEL ACTIONS:")
    for action, count in action_counts.items():
        percentage = (count / len(actions)) * 100 if actions else 0
        print(f"  {action}: {count} ({percentage:.1f}%)")
    
    return action_counts

if enhanced_recommendations:
    enhanced_actions = analyze_actions(enhanced_recommendations, "ENHANCED")
    
if base_recommendations:
    base_actions = analyze_actions(base_recommendations, "BASE")
    
if actual_recommendations:
    actual_actions = analyze_actions(actual_recommendations, "ACTUAL")

# Confidence analysis
if enhanced_recommendations:
    enhanced_confidences = [rec.get('confidence', 0) for rec in enhanced_recommendations if rec.get('confidence') is not None]
    if enhanced_confidences:
        print(f"\n📈 ENHANCED MODEL CONFIDENCE:")
        print(f"  Mean: {np.mean(enhanced_confidences):.1f}%")
        print(f"  Median: {np.median(enhanced_confidences):.1f}%")
        print(f"  Range: {np.min(enhanced_confidences):.1f}% - {np.max(enhanced_confidences):.1f}%")

if base_recommendations:
    base_confidences = [rec.get('confidence', 0) for rec in base_recommendations if rec.get('confidence') is not None]
    if base_confidences:
        print(f"\n📈 BASE MODEL CONFIDENCE:")
        print(f"  Mean: {np.mean(base_confidences):.1f}%")
        print(f"  Median: {np.median(base_confidences):.1f}%")
        print(f"  Range: {np.min(base_confidences):.1f}% - {np.max(base_confidences):.1f}%")

# Action accuracy comparison
action_matches = 0
total_comparisons = 0

for result in results:
    enhanced_rec = result.get('enhanced_recommendation')
    actual_rec = result.get('actual_recommendation')
    
    if enhanced_rec and actual_rec:
        enhanced_action = enhanced_rec.get('action')
        actual_action = actual_rec.get('action')
        
        if enhanced_action and actual_action:
            total_comparisons += 1
            if enhanced_action == actual_action:
                action_matches += 1

if total_comparisons > 0:
    action_accuracy = (action_matches / total_comparisons) * 100
    print(f"\n🎯 ACTION ACCURACY:")
    print(f"  Enhanced model action accuracy: {action_accuracy:.1f}% ({action_matches}/{total_comparisons})")

# Sample recommendations display
print(f"\n📋 SAMPLE TRADING RECOMMENDATIONS:")
print("-" * 80)

for i, result in enumerate(results[:5]):  # Show first 5 samples
    print(f"\nSample {result['sample_id']+1}:")
    
    enhanced_rec = result.get('enhanced_recommendation')
    if enhanced_rec:
        print(f"  Enhanced: {enhanced_rec.get('action', 'N/A')} | "
              f"Confidence: {enhanced_rec.get('confidence', 'N/A')}% | "
              f"Stop Loss: ${enhanced_rec.get('stop_loss', 'N/A')} | "
              f"Take Profit: ${enhanced_rec.get('take_profit', 'N/A')}")
    
    base_rec = result.get('base_recommendation')
    if base_rec:
        print(f"  Base:     {base_rec.get('action', 'N/A')} | "
              f"Confidence: {base_rec.get('confidence', 'N/A')}% | "
              f"Stop Loss: ${base_rec.get('stop_loss', 'N/A')} | "
              f"Take Profit: ${base_rec.get('take_profit', 'N/A')}")
    
    actual_rec = result.get('actual_recommendation')
    if actual_rec:
        print(f"  Actual:   {actual_rec.get('action', 'N/A')} | "
              f"Confidence: {actual_rec.get('confidence', 'N/A')}% | "
              f"Stop Loss: ${actual_rec.get('stop_loss', 'N/A')} | "
              f"Take Profit: ${actual_rec.get('take_profit', 'N/A')}")
    
    print(f"  Forecast: Enhanced {result['enhanced_last_10'][:3]}... | "
          f"Base {result['base_last_10'][:3]}... | "
          f"Actual {result['actual_last_10'][:3]}...")

## Analysis of Last 10 Prices

In [ ]:
# Analyze the last 10 prices
valid_comparisons = []

for result in results:
    if (len(result['enhanced_last_10']) >= 5 and 
        len(result['base_last_10']) >= 5 and 
        len(result['actual_last_10']) >= 5):
        valid_comparisons.append(result)

print(f"Found {len(valid_comparisons)} samples with valid last 10 price predictions\n")

if valid_comparisons:
    # Calculate statistics for last 10 prices
    enhanced_last_10_all = []
    base_last_10_all = []
    actual_last_10_all = []
    
    for comp in valid_comparisons:
        enhanced_last_10_all.extend(comp['enhanced_last_10'])
        base_last_10_all.extend(comp['base_last_10'])
        actual_last_10_all.extend(comp['actual_last_10'])
    
    print("=" * 80)
    print("📊 LAST 10 PRICES STATISTICS")
    print("=" * 80)
    
    # Enhanced model stats
    print(f"\n🔥 ENHANCED MODEL LAST 10 PRICES:")
    print(f"  Mean: ${np.mean(enhanced_last_10_all):.2f}")
    print(f"  Median: ${np.median(enhanced_last_10_all):.2f}")
    print(f"  Std: ${np.std(enhanced_last_10_all):.2f}")
    print(f"  Min: ${np.min(enhanced_last_10_all):.2f}")
    print(f"  Max: ${np.max(enhanced_last_10_all):.2f}")
    print(f"  Total predictions: {len(enhanced_last_10_all)}")
    
    # Base model stats
    print(f"\n⚡ BASE MODEL LAST 10 PRICES:")
    print(f"  Mean: ${np.mean(base_last_10_all):.2f}")
    print(f"  Median: ${np.median(base_last_10_all):.2f}")
    print(f"  Std: ${np.std(base_last_10_all):.2f}")
    print(f"  Min: ${np.min(base_last_10_all):.2f}")
    print(f"  Max: ${np.max(base_last_10_all):.2f}")
    print(f"  Total predictions: {len(base_last_10_all)}")
    
    # Actual prices stats
    print(f"\n🎯 ACTUAL LAST 10 PRICES:")
    print(f"  Mean: ${np.mean(actual_last_10_all):.2f}")
    print(f"  Median: ${np.median(actual_last_10_all):.2f}")
    print(f"  Std: ${np.std(actual_last_10_all):.2f}")
    print(f"  Min: ${np.min(actual_last_10_all):.2f}")
    print(f"  Max: ${np.max(actual_last_10_all):.2f}")
    print(f"  Total actual prices: {len(actual_last_10_all)}")
    
    # Sample-by-sample comparison
    print(f"\n📋 SAMPLE-BY-SAMPLE LAST 10 PRICES COMPARISON:")
    print("-" * 80)
    
    for i, comp in enumerate(valid_comparisons[:10]):  # Show first 10 samples
        print(f"\nSample {comp['sample_id']+1}:")
        print(f"  Enhanced: {[f'${p:.2f}' for p in comp['enhanced_last_10']]}")
        print(f"  Base:     {[f'${p:.2f}' for p in comp['base_last_10']]}")
        print(f"  Actual:   {[f'${p:.2f}' for p in comp['actual_last_10']]}")
        
        # Calculate differences
        if len(comp['actual_last_10']) > 0:
            min_len = min(len(comp['enhanced_last_10']), len(comp['actual_last_10']))
            if min_len > 0:
                enhanced_diff = np.mean(np.abs(np.array(comp['enhanced_last_10'][:min_len]) - np.array(comp['actual_last_10'][:min_len])))
                base_diff = np.mean(np.abs(np.array(comp['base_last_10'][:min_len]) - np.array(comp['actual_last_10'][:min_len])))
                print(f"  Enhanced MAE: ${enhanced_diff:.2f}")
                print(f"  Base MAE:     ${base_diff:.2f}")
                print(f"  Better model: {'Enhanced' if enhanced_diff < base_diff else 'Base'}")

else:
    print("❌ No valid comparisons found with sufficient last 10 price predictions")

## Visualization of Last 10 Prices

In [ ]:
if valid_comparisons:
    # Create visualizations including trading recommendations
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    fig.suptitle('Bitcoin Trading Recommendations & Last 10 Price Predictions Comparison', fontsize=16, fontweight='bold')
    
    # 1. Distribution comparison of last 10 prices
    axes[0,0].hist([enhanced_last_10_all, base_last_10_all, actual_last_10_all], 
                   bins=30, alpha=0.7, 
                   label=['Enhanced Model', 'Base Model', 'Actual Prices'], 
                   color=['blue', 'red', 'green'])
    axes[0,0].set_title('Distribution of Last 10 Price Predictions', fontweight='bold')
    axes[0,0].set_xlabel('Price ($)')
    axes[0,0].set_ylabel('Frequency')
    axes[0,0].legend()
    axes[0,0].grid(True, alpha=0.3)
    
    # 2. Box plot comparison
    data_for_boxplot = [enhanced_last_10_all, base_last_10_all, actual_last_10_all]
    labels_for_boxplot = ['Enhanced', 'Base', 'Actual']
    axes[0,1].boxplot(data_for_boxplot, labels=labels_for_boxplot)
    axes[0,1].set_title('Last 10 Prices Box Plot Comparison', fontweight='bold')
    axes[0,1].set_ylabel('Price ($)')
    axes[0,1].grid(True, alpha=0.3)
    
    # 3. Action distribution comparison
    enhanced_actions_for_plot = [result.get('enhanced_recommendation', {}).get('action', 'UNKNOWN') for result in valid_comparisons]
    base_actions_for_plot = [result.get('base_recommendation', {}).get('action', 'UNKNOWN') for result in valid_comparisons]
    actual_actions_for_plot = [result.get('actual_recommendation', {}).get('action', 'UNKNOWN') for result in valid_comparisons]
    
    all_actions = list(set(enhanced_actions_for_plot + base_actions_for_plot + actual_actions_for_plot))
    enhanced_counts = [enhanced_actions_for_plot.count(action) for action in all_actions]
    base_counts = [base_actions_for_plot.count(action) for action in all_actions]
    actual_counts = [actual_actions_for_plot.count(action) for action in all_actions]
    
    x = np.arange(len(all_actions))
    width = 0.25
    
    axes[0,2].bar(x - width, enhanced_counts, width, label='Enhanced', alpha=0.8, color='blue')
    axes[0,2].bar(x, base_counts, width, label='Base', alpha=0.8, color='red')
    axes[0,2].bar(x + width, actual_counts, width, label='Actual', alpha=0.8, color='green')
    
    axes[0,2].set_title('Trading Action Distribution', fontweight='bold')
    axes[0,2].set_xlabel('Action')
    axes[0,2].set_ylabel('Count')
    axes[0,2].set_xticks(x)
    axes[0,2].set_xticklabels(all_actions)
    axes[0,2].legend()
    axes[0,2].grid(True, alpha=0.3)
    
    # 4. Confidence comparison
    enhanced_confidences_plot = [result.get('enhanced_recommendation', {}).get('confidence', 0) 
                                for result in valid_comparisons if result.get('enhanced_recommendation', {}).get('confidence') is not None]
    base_confidences_plot = [result.get('base_recommendation', {}).get('confidence', 0) 
                            for result in valid_comparisons if result.get('base_recommendation', {}).get('confidence') is not None]
    
    if enhanced_confidences_plot and base_confidences_plot:
        axes[1,0].hist([enhanced_confidences_plot, base_confidences_plot], 
                       bins=20, alpha=0.7, 
                       label=['Enhanced Model', 'Base Model'], 
                       color=['blue', 'red'])
        axes[1,0].set_title('Confidence Distribution', fontweight='bold')
        axes[1,0].set_xlabel('Confidence (%)')
        axes[1,0].set_ylabel('Frequency')
        axes[1,0].legend()
        axes[1,0].grid(True, alpha=0.3)
    
    # 5. Sample-wise comparison for first few samples
    sample_indices = list(range(min(5, len(valid_comparisons))))
    enhanced_means = [np.mean(valid_comparisons[i]['enhanced_last_10']) for i in sample_indices]
    base_means = [np.mean(valid_comparisons[i]['base_last_10']) for i in sample_indices]
    actual_means = [np.mean(valid_comparisons[i]['actual_last_10']) for i in sample_indices]
    
    x = np.arange(len(sample_indices))
    width = 0.25
    
    axes[1,1].bar(x - width, enhanced_means, width, label='Enhanced', alpha=0.8, color='blue')
    axes[1,1].bar(x, base_means, width, label='Base', alpha=0.8, color='red')
    axes[1,1].bar(x + width, actual_means, width, label='Actual', alpha=0.8, color='green')
    
    axes[1,1].set_title('Mean of Last 10 Prices by Sample', fontweight='bold')
    axes[1,1].set_xlabel('Sample Index')
    axes[1,1].set_ylabel('Mean Price ($)')
    axes[1,1].set_xticks(x)
    axes[1,1].set_xticklabels([f'S{valid_comparisons[i]["sample_id"]+1}' for i in sample_indices])
    axes[1,1].legend()
    axes[1,1].grid(True, alpha=0.3)
    
    # 6. Price trend for a specific sample
    if len(valid_comparisons) > 0:
        sample_to_plot = valid_comparisons[0]
        days = list(range(1, len(sample_to_plot['enhanced_last_10']) + 1))
        
        axes[1,2].plot(days, sample_to_plot['enhanced_last_10'], 'o-', label='Enhanced', color='blue', linewidth=2)
        axes[1,2].plot(days, sample_to_plot['base_last_10'][:len(days)], 's-', label='Base', color='red', linewidth=2)
        axes[1,2].plot(days, sample_to_plot['actual_last_10'][:len(days)], '^-', label='Actual', color='green', linewidth=2)
        
        # Add action and confidence info to title
        enhanced_rec = sample_to_plot.get('enhanced_recommendation', {})
        title_text = f'Price Trend - Sample {sample_to_plot["sample_id"]+1}'
        if enhanced_rec.get('action'):
            title_text += f' (Enhanced: {enhanced_rec.get("action")} @ {enhanced_rec.get("confidence", "N/A")}%)'
        
        axes[1,2].set_title(title_text, fontweight='bold')
        axes[1,2].set_xlabel('Day')
        axes[1,2].set_ylabel('Price ($)')
        axes[1,2].legend()
        axes[1,2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('trading_recommendations_and_last_10_prices_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("📊 Trading recommendations and last 10 prices comparison visualization saved as 'trading_recommendations_and_last_10_prices_comparison.png'")

else:
    print("❌ Cannot create visualizations - insufficient data")

## Detailed Last 10 Prices Analysis

In [ ]:
if valid_comparisons:
    # Calculate MAE for last 10 prices only
    enhanced_mae_last_10 = []
    base_mae_last_10 = []
    
    for comp in valid_comparisons:
        # Enhanced vs Actual
        min_len_enhanced = min(len(comp['enhanced_last_10']), len(comp['actual_last_10']))
        if min_len_enhanced > 0:
            enhanced_mae = np.mean(np.abs(np.array(comp['enhanced_last_10'][:min_len_enhanced]) - 
                                        np.array(comp['actual_last_10'][:min_len_enhanced])))
            enhanced_mae_last_10.append(enhanced_mae)
        
        # Base vs Actual
        min_len_base = min(len(comp['base_last_10']), len(comp['actual_last_10']))
        if min_len_base > 0:
            base_mae = np.mean(np.abs(np.array(comp['base_last_10'][:min_len_base]) - 
                                    np.array(comp['actual_last_10'][:min_len_base])))
            base_mae_last_10.append(base_mae)
    
    print("=" * 80)
    print("🎯 LAST 10 PRICES ACCURACY ANALYSIS")
    print("=" * 80)
    
    if enhanced_mae_last_10 and base_mae_last_10:
        print(f"\n📊 Mean Absolute Error (MAE) for Last 10 Prices:")
        print(f"  Enhanced Model MAE: ${np.mean(enhanced_mae_last_10):.2f} (±${np.std(enhanced_mae_last_10):.2f})")
        print(f"  Base Model MAE:     ${np.mean(base_mae_last_10):.2f} (±${np.std(base_mae_last_10):.2f})")
        
        improvement = ((np.mean(base_mae_last_10) - np.mean(enhanced_mae_last_10)) / np.mean(base_mae_last_10)) * 100
        print(f"  Improvement: {improvement:.2f}%")
        
        # Count how many times enhanced model is better
        enhanced_better_count = sum(1 for e, b in zip(enhanced_mae_last_10, base_mae_last_10) if e < b)
        print(f"\n🏆 Enhanced model is more accurate in {enhanced_better_count}/{len(enhanced_mae_last_10)} samples ({enhanced_better_count/len(enhanced_mae_last_10)*100:.1f}%)")
    
        # Save detailed results including trading recommendations
    detailed_results = {
        'summary': {
            'total_samples_analyzed': len(valid_comparisons),
            'enhanced_model_mae_last_10': float(np.mean(enhanced_mae_last_10)) if enhanced_mae_last_10 else None,
            'base_model_mae_last_10': float(np.mean(base_mae_last_10)) if base_mae_last_10 else None,
            'improvement_percentage': float(improvement) if enhanced_mae_last_10 and base_mae_last_10 else None,
            'enhanced_better_count': enhanced_better_count if enhanced_mae_last_10 and base_mae_last_10 else None,
            'trading_recommendations_summary': {
                'enhanced_actions': enhanced_actions if 'enhanced_actions' in locals() else {},
                'base_actions': base_actions if 'base_actions' in locals() else {},
                'actual_actions': actual_actions if 'actual_actions' in locals() else {},
                'action_accuracy': action_accuracy if 'action_accuracy' in locals() else None
            }
        },
        'detailed_comparisons': []
    }
    
    for comp in valid_comparisons:
        comparison_data = {
            'sample_id': comp['sample_id'],
            'enhanced_last_10': [float(p) for p in comp['enhanced_last_10']],
            'base_last_10': [float(p) for p in comp['base_last_10']],
            'actual_last_10': [float(p) for p in comp['actual_last_10']],
            'enhanced_generated_text': comp['enhanced_text'],
            'base_generated_text': comp['base_text']
        }
        
        # Add trading recommendation data if available
        if comp.get('enhanced_recommendation'):
            comparison_data['enhanced_recommendation'] = comp['enhanced_recommendation']
        if comp.get('base_recommendation'):
            comparison_data['base_recommendation'] = comp['base_recommendation']
        if comp.get('actual_recommendation'):
            comparison_data['actual_recommendation'] = comp['actual_recommendation']
            
        detailed_results['detailed_comparisons'].append(comparison_data)
    
    with open('trading_recommendations_and_last_10_prices_analysis.json', 'w') as f:
        json.dump(detailed_results, f, indent=2)
    
    print(f"\\n💾 Detailed trading recommendations and last 10 prices analysis saved to 'trading_recommendations_and_last_10_prices_analysis.json'")

else:
    print("❌ No valid data for detailed analysis")

## Show Raw Model Outputs for Last Few Samples

In [ ]:
if results:
    print("🔍 RAW MODEL OUTPUTS (Last 3 Samples):")
    print("=" * 100)
    
    for i, result in enumerate(results[-3:]):  # Show last 3 samples
        print(f"\n📋 SAMPLE {result['sample_id']+1}:")
        print("-" * 50)
        
        print(f"\n🔥 ENHANCED MODEL OUTPUT:")
        print(f"Text: {result['enhanced_text']}")
        print(f"Extracted Prices: {result['enhanced_full']}")
        print(f"Last 10: {result['enhanced_last_10']}")
        
        print(f"\n⚡ BASE MODEL OUTPUT:")
        print(f"Text: {result['base_text']}")
        print(f"Extracted Prices: {result['base_full']}")
        print(f"Last 10: {result['base_last_10']}")
        
        print(f"\n🎯 ACTUAL OUTPUT:")
        print(f"Extracted Prices: {result['actual_full']}")
        print(f"Last 10: {result['actual_last_10']}")
        
        print("=" * 100)

else:
    print("❌ No results to display")